In [ ]:
# ==============================================================
# TRANSFER LEARNING FOR MULTI-LABEL MEDICAL IMAGE CLASSIFICATION
# Deep Learning Fall 2025
# Supervised by: Dr. Qing Liu
# Team: Fahad Ibne Fahian, Shane Dalumura Hettige
# ==============================================================

import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score

# ========================
# Task 2: LOSS FUNCTIONS
# ========================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        p = torch.sigmoid(inputs)
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        p_t = torch.where(targets == 1, p, 1 - p)
        focal_weight = (1 - p_t) ** self.gamma
        focal_loss = self.alpha * focal_weight * bce_loss
        return focal_loss.mean()

class ClassBalancedLoss(nn.Module):
    def __init__(self, samples_per_class=None, beta=0.9999):
        super(ClassBalancedLoss, self).__init__()
        self.samples_per_class = samples_per_class
        self.beta = beta
        self.weights = None
        if samples_per_class is not None:
            self._calculate_weights()

    def _calculate_weights(self):
        if self.samples_per_class is None: return
        effective_num = 1.0 - torch.pow(self.beta, torch.tensor(self.samples_per_class, dtype=torch.float32))
        weights = (1.0 - self.beta) / effective_num
        weights = weights / weights.sum() * len(weights)
        try:
            self.register_buffer('weights', weights)
        except KeyError:
            self.weights = weights

    def forward(self, inputs, targets, samples_per_class=None):
        if self.weights is not None:
            bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
            weighted_loss = bce_loss * self.weights.to(inputs.device).unsqueeze(0)
            return weighted_loss.mean()
        else:
            return F.binary_cross_entropy_with_logits(inputs, targets)

class HybridLoss(nn.Module):
    def __init__(self, samples_per_class=None, alpha=0.25, gamma=2.0, beta=0.9999, lambda_weight=0.5):
        super(HybridLoss, self).__init__()
        self.focal_loss = FocalLoss(alpha=alpha, gamma=gamma)
        self.class_balanced_loss = ClassBalancedLoss(samples_per_class=samples_per_class, beta=beta)
        self.lambda_weight = lambda_weight

    def forward(self, inputs, targets):
        focal = self.focal_loss(inputs, targets)
        class_balanced = self.class_balanced_loss(inputs, targets)
        return self.lambda_weight * focal + (1 - self.lambda_weight) * class_balanced

# ========================
# Task 3: ATTENTION MECHANISMS
# ========================
class SqueezeExcitation(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(SqueezeExcitation, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, h, w = x.size()
        squeeze = F.adaptive_avg_pool2d(x, 1).view(b, c)
        excitation = self.fc(squeeze).view(b, c, 1, 1)
        return x * excitation

class MultiHeadAttention(nn.Module):
    def __init__(self, in_channels, num_heads=4):
        super(MultiHeadAttention, self).__init__()
        assert in_channels % num_heads == 0, "in_channels must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = in_channels // num_heads
        self.query_conv = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.key_conv = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.value_conv = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.out_conv = nn.Conv2d(in_channels, in_channels, kernel_size=1)

    def forward(self, x):
        b, c, h, w = x.size()
        queries = self.query_conv(x).view(b, self.num_heads, self.head_dim, h * w)
        keys = self.key_conv(x).view(b, self.num_heads, self.head_dim, h * w)
        values = self.value_conv(x).view(b, self.num_heads, self.head_dim, h * w)
        scores = torch.matmul(queries.transpose(-2, -1), keys) / (self.head_dim ** 0.5)
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, values.transpose(-2, -1)).transpose(-2, -1).contiguous()
        out = out.view(b, c, h, w)
        return self.out_conv(out)

# ========================
# MODELS & DATASET
# ========================
class RetinaMultiLabelDataset(Dataset):
    def __init__(self, csv_file, image_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_name = row.iloc[0]
        img_path = os.path.join(self.image_dir, img_name)
        try:
            img = Image.open(img_path).convert("RGB")
        except:
            img = Image.new('RGB', (256, 256)) # Fallback
        
        labels = torch.tensor(row[1:4].values.astype("float32"))
        if self.transform: img = self.transform(img)
        return img, labels, img_name

def build_model(backbone="resnet18", num_classes=3, pretrained=True, attention_type="mha"):
    if backbone == "resnet18":
        model = models.resnet18(pretrained=pretrained)
        # Inject Attention
        if attention_type == "se":
            model.layer4[1].se = SqueezeExcitation(512)
        elif attention_type == "mha":
            model.layer4[1].mha = MultiHeadAttention(512, num_heads=4)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif backbone == "efficientnet":
        model = models.efficientnet_b0(pretrained=pretrained)
        if attention_type == "se":
            model.features[8].se = SqueezeExcitation(320)
        elif attention_type == "mha":
            model.features[8].mha = MultiHeadAttention(320, num_heads=4)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model

# ========================
# TRAINING PIPELINE
# ========================
def train_one_backbone(backbone, train_csv, val_csv, test_csv, onsite_csv,
                       train_image_dir, val_image_dir, test_image_dir, onsite_image_dir,
                       epochs=10, batch_size=32, lr=1e-4, img_size=256, save_dir="checkpoints", 
                       pretrained_backbone=None, loss_type="hybrid", attention_type="mha"):
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device} | Loss: {loss_type} | Attention: {attention_type}")

    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # Load Data
    train_ds = RetinaMultiLabelDataset(train_csv, train_image_dir, transform)
    val_ds = RetinaMultiLabelDataset(val_csv, val_image_dir, transform)
    test_ds = RetinaMultiLabelDataset(test_csv, test_image_dir, transform)
    onsite_ds = RetinaMultiLabelDataset(onsite_csv, onsite_image_dir, transform)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    onsite_loader = DataLoader(onsite_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    # Initialize Model
    model = build_model(backbone, num_classes=3, pretrained=False, attention_type=attention_type).to(device)
    
    # Define Loss
    if loss_type == "focal":
        criterion = FocalLoss(alpha=0.25, gamma=2.0)
    elif loss_type == "class_balanced":
        samples_per_class = pd.read_csv(train_csv).iloc[:, 1:4].sum().values.tolist()
        criterion = ClassBalancedLoss(samples_per_class=samples_per_class)
    elif loss_type == "hybrid":
        samples_per_class = pd.read_csv(train_csv).iloc[:, 1:4].sum().values.tolist()
        criterion = HybridLoss(samples_per_class=samples_per_class)
    else:
        criterion = nn.BCEWithLogitsLoss()

    optimizer = optim.Adam(model.parameters(), lr=lr)

    # Load Pretrained Weights
    if pretrained_backbone and os.path.exists(pretrained_backbone):
        print(f"Loading backbone from {pretrained_backbone}")
        try:
            state_dict = torch.load(pretrained_backbone, map_location=device)
            if 'state_dict' in state_dict: state_dict = state_dict['state_dict']
            model.load_state_dict(state_dict, strict=False)
        except Exception as e: print(f"Error loading backbone: {e}")

    # Training Loop
    best_f1 = 0.0
    os.makedirs(save_dir, exist_ok=True)
    ckpt_path = os.path.join(save_dir, f"Fahad_task3.pt")

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for imgs, labels, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * imgs.size(0)
        print(f"Epoch {epoch+1} Loss: {train_loss/len(train_loader.dataset):.4f}")

        # Validation (Save best based on F1)
        model.eval()
        y_true, y_pred = [], []
        with torch.no_grad():
            for imgs, labels, _ in val_loader:
                imgs = imgs.to(device)
                outputs = model(imgs)
                probs = torch.sigmoid(outputs).cpu().numpy()
                preds = (probs > 0.5).astype(int)
                y_true.extend(labels.numpy())
                y_pred.extend(preds)
        
        f1 = f1_score(np.array(y_true), np.array(y_pred), average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), ckpt_path)
            print(f"New Best F1: {best_f1:.4f} | Model Saved!")

    # Final Offsite Evaluation
    print("\n>>> Evaluating on Offsite Test Set...")
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for imgs, labels, _ in test_loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            preds = (torch.sigmoid(outputs).cpu().numpy() > 0.5).astype(int)
            y_true.extend(labels.numpy())
            y_pred.extend(preds)
            
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    for i, name in enumerate(["DR", "Glaucoma", "AMD"]):
        f1 = f1_score(y_true[:, i], y_pred[:, i], zero_division=0)
        print(f"{name} F1: {f1:.4f}")
    print(f"Average F1: {f1_score(y_true, y_pred, average='macro'):.4f}")

    # Onsite Submission
    print("\n>>> Generating Submission...")
    results = []
    with torch.no_grad():
        for imgs, _, names in tqdm(onsite_loader):
            imgs = imgs.to(device)
            preds = (torch.sigmoid(model(imgs)).cpu().numpy() > 0.5).astype(int)
            for i in range(len(names)):
                results.append([names[i], preds[i][0], preds[i][1], preds[i][2]])
    
    pd.DataFrame(results, columns=['id', 'D', 'G', 'A']).to_csv("submission.csv", index=False)
    print("submission.csv saved!")

if __name__ == "__main__":
    # KAGGLE PATHS (Edit if running locally). For Colab, use: base_path = "/content"
    base_path = "/kaggle/input/odir-dataset"
    if not os.path.exists(base_path): base_path = "." # Fallback to local
    
    train_one_backbone(
        backbone='resnet18',
        train_csv=f"{base_path}/train.csv",
        val_csv=f"{base_path}/val.csv",
        test_csv=f"{base_path}/offsite_test.csv",
        onsite_csv=f"{base_path}/onsite_test_submission.csv",
        train_image_dir=f"{base_path}/Images/train" if os.path.exists(base_path) else "./Images/train",
        val_image_dir=f"{base_path}/Images/val" if os.path.exists(base_path) else "./Images/val",
        test_image_dir=f"{base_path}/Images/offsite_test" if os.path.exists(base_path) else "./Images/offsite_test",
        onsite_image_dir=f"{base_path}/Images/onsite_test" if os.path.exists(base_path) else "./Images/onsite_test",
        epochs=15, 
        batch_size=32,
        pretrained_backbone=f"{base_path}/pretrained_backbone/ckpt_resnet18_ep50.pt" if os.path.exists(base_path) else None,
        loss_type="hybrid",      # Use Hybrid (Focal + Class Balanced)
        attention_type="mha"     # Use Multi-Head Attention
    )